# Chapter 3 - Instrumental Variables

<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#3.2.3-Simulation-of-Confounded-Data" data-toc-modified-id="3.2.3-Simulation-of-Confounded-Data-1">3.2.3 Simulation of Confounded Data</a></span></li><li><span><a href="#3.3.4-Simulation-of-an-IV-Estimator" data-toc-modified-id="3.3.4-Simulation-of-an-IV-Estimator-2">3.3.4 Simulation of an IV Estimator</a></span></li><li><span><a href="#3.3.7-IV-Estimator-in-Python" data-toc-modified-id="3.3.7-IV-Estimator-in-Python-3">3.3.7 IV Estimator in Python</a></span></li><li><span><a href="#3.3.8-Bootstrap-IV-Estimator-for-Python" data-toc-modified-id="3.3.8-Bootstrap-IV-Estimator-for-Python-4">3.3.8 Bootstrap IV Estimator for Python</a></span></li><li><span><a href="#3.4.2-NLSM-Data" data-toc-modified-id="3.4.2-NLSM-Data-5">3.4.2 NLSM Data</a></span></li><li><span><a href="#3.4.3-Simple-IV-Estimates-of-Returns-to-Schooling" data-toc-modified-id="3.4.3-Simple-IV-Estimates-of-Returns-to-Schooling-6">3.4.3 Simple IV Estimates of Returns to Schooling</a></span></li><li><span><a href="#3.4.4-Matrix-Algebra-IV-Estimates-of-Returns-to-Schooling" data-toc-modified-id="3.4.4-Matrix-Algebra-IV-Estimates-of-Returns-to-Schooling-7">3.4.4 Matrix Algebra IV Estimates of Returns to Schooling</a></span></li><li><span><a href="#3.4.5-Concerns-with-Distance-to-College" data-toc-modified-id="3.4.5-Concerns-with-Distance-to-College-8">3.4.5 Concerns with Distance to College</a></span></li><li><span><a href="#3.5.2-Test-of-Instrument-Validity-in-Python" data-toc-modified-id="3.5.2-Test-of-Instrument-Validity-in-Python-9">3.5.2 Test of Instrument Validity in Python</a></span></li><li><span><a href="#3.6.4-LATE-Estimates-of-Returns-to-Schooling" data-toc-modified-id="3.6.4-LATE-Estimates-of-Returns-to-Schooling-10">3.6.4 LATE Estimates of Returns to Schooling</a></span></li></ul></div>

## 3.2.3 Simulation of Confounded Data

In [1]:
# as in chapters 1 and 2, we use numpy
import numpy as np

In [2]:
# remove Python warnings
import warnings
warnings.filterwarnings("ignore")

In [3]:
# setting the seed in Numpy
# the random numbers of Python are not the ones of R,
# so our results are close to the book but not the same
np.random.seed(123456789)

# main parameters of interest
N = 1000
a = 2
b = 3
c = 2
e = 3
f = -1
d = 4

# the instrument z is drawn from a uniform distribution
z = np.random.rand(N)

# two vectors of unobserved characteristics, from normal distributions
# loc is the mean and scale is the standard deviation (sd in R)
u_1 = np.random.normal(loc=0, scale=3, size=N)
u_2 = np.random.normal(loc=0, scale=1, size=N)

# x depends on the instrument z and on the unobserved u_1
x = f + d*z + u_2 + c*u_1

# y depends on x and on the same u_1
# this is the problem: u_1 moves both x and y
y = a + b*x + e*u_1

We run the OLS regression of y on x:

In [4]:
# using statsmodels and pandas, as in chapter 2
import statsmodels.formula.api as sm
from statsmodels.iolib.summary2 import summary_col
import pandas as pd

# pack the variables in a dataframe
data = pd.DataFrame({'y': y, 'x': x, 'z': z})

lm1 = sm.ols(formula="y ~ x", data=data).fit()

print(summary_col([lm1],
                  stars=True,
                  float_format='%0.2f',
                  model_names=['(1)'],
                  info_dict={'N':lambda x: "{0:d}".format(int(x.nobs)),
                             'R2':lambda x: "{:.2f}".format(x.rsquared)}
            ))


                 (1)  
----------------------
Intercept      0.57***
               (0.07) 
x              4.41***
               (0.01) 
R-squared      0.99   
R-squared Adj. 0.99   
N              1000   
R2             0.99   
Standard errors in
parentheses.
* p<.1, ** p<.05,
***p<.01


In [5]:
# the true value of b is 3
# the OLS estimate of b is too high
print(lm1.params['x'])

4.412745800831298


## 3.3.4 Simulation of an IV Estimator

In [6]:
# regression of y on the instrument z (the slope is b times d)
# picking the slope coefficient from each regression
bd_hat = sm.ols(formula="y ~ z", data=data).fit().params['z']

# regression of x on the instrument z (the slope is d)
d_hat = sm.ols(formula="x ~ z", data=data).fit().params['z']

# the ratio of the two slopes is the IV estimate of b
print(bd_hat/d_hat)

2.741793253756314


## 3.3.7 IV Estimator in Python

In [7]:
# in python the inverse matrix can be found using
# the inv() function (from Numpy)
from numpy.linalg import inv

# remember the column of 1's for the intercept
X = np.c_[np.ones(N), x]
# remember Z same size as X
Z = np.c_[np.ones(N), z]

# OLS estimator
# .T to transpose a matrix and .dot() to multiply two matrices
beta_hat_ols = inv(X.T.dot(X)).dot(X.T).dot(y)

# IV estimator
# same formula, but the transpose of Z replaces the transpose of X
beta_hat_iv = inv(Z.T.dot(X)).dot(Z.T).dot(y)

In [8]:
# first number is the intercept (true value 2)
# second number is the slope (true value 3)
beta_hat_ols

array([0.57331195, 4.4127458 ])

In [9]:
beta_hat_iv

array([2.12785508, 2.74179325])

## 3.3.8 Bootstrap IV Estimator for Python

In [10]:
def lm_iv(y, X_in, Z_in=None, Reps=100, min_in=0.05, max_in=0.95):
    """
    Bootstrap IV estimator
    * takes in the y variable, the x explanatory variables
      and the z variables if available
    * without z variables, the x variables are used in place of z
      and the result is the OLS estimator
    * the "=" creates a default value for the local variables
    """

    # set up
    np.random.seed(123456789)
    if Z_in is None:
        Z_in = X_in
    N = len(y)  # number of observations
    X = np.c_[np.ones(N), X_in]  # adds a column of 1's to the matrix
    Z = np.c_[np.ones(N), Z_in]

    # bootstrap
    # create an empty list to fill with the estimates
    rows = []
    for r in range(0, Reps):
        # we draw N row numbers at random, with replacement
        # python starts counting at 0, so the rows go from 0 to N-1
        index_bs = np.random.randint(0, N, size=N)
        y_bs = y[index_bs]  # note y is a vector
        X_bs = X[index_bs, :]
        Z_bs = Z[index_bs, :]
        rows.append(inv(Z_bs.T.dot(X_bs)).dot(Z_bs.T).dot(y_bs))

    # stacking all the estimates in a single dataframe
    # one row per bootstrap sample, one column per coefficient
    bs_mat = pd.DataFrame(rows)

    # present results
    tab_res = pd.DataFrame()
    tab_res['coef'] = bs_mat.mean()
    tab_res['sd'] = bs_mat.std()
    tab_res[str(min_in)] = bs_mat.quantile(min_in)
    tab_res[str(max_in)] = bs_mat.quantile(max_in)

    # returns the tab_res dataframe
    return tab_res

In [11]:
# OLS
# row 0 is the intercept and row 1 is the slope
lm_iv(y, x)

,coef,sd,0.05,0.95
0,0.571382,0.080959,0.441124,0.697039
1,4.412921,0.010955,4.396797,4.430975


In [12]:
# IV
lm_iv(y, x, z)

,coef,sd,0.05,0.95
0,2.159181,0.486213,1.589501,2.935924
1,2.687903,0.473977,1.935627,3.156152


## 3.4.2 NLSM Data

In [13]:
x = pd.read_csv('../data/nls.csv')

In [14]:
# we take the cleaning functions of chapter 2

def convert_col_to_numeric(df, col):
    # values which are not numbers (here ".") become NAs
    df[col] = pd.to_numeric(df[col], errors = 'coerce')
    return df

def remove_missing(df, col):
    condition = df[col].isna() == False
    return df[condition]

def get_working_years_after_school(df):
    """
    We create new variables:
    * exp: working years after school
    * exp2: experience squared divided by 100
    """
    df['exp'] = df['age76']-df['ed76']-6
    df['exp2'] = df['exp']**2/100
    return df

In [15]:
# and we perform the data cleaning using the .pipe operator
# on the original dataset x

x1 = (
    x
    .pipe(convert_col_to_numeric, col = 'lwage76')
    .pipe(remove_missing, col = 'lwage76')
    .pipe(get_working_years_after_school)
)

## 3.4.3 Simple IV Estimates of Returns to Schooling

In [16]:
# OLS estimate
# smsa refers to urban or rural, with 76 refering to 1976.
# reg refers to region of the US - North, South, West etc.
# 66 refers to 1966.
lm4 = sm.ols(formula = "lwage76 ~ ed76 + exp + exp2 + black + reg76r + smsa76r + smsa66r + reg662 + reg663 + reg664 + reg665 + reg666 + reg667 + reg668 + reg669", data=x1).fit()
print(lm4.params['ed76'])

0.0746932594020458


In [17]:
# Intent-To-Treat estimate
# same regression, but the instrument nearc4 replaces ed76
# nearc4 is a dummy for distance to a 4 year college.
lm5 = sm.ols(formula = "lwage76 ~ nearc4 + exp + exp2 + black + reg76r + smsa76r + smsa66r + reg662 + reg663 + reg664 + reg665 + reg666 + reg667 + reg668 + reg669", data=x1).fit()
print(lm5.params['nearc4'])

0.04206793440995808


In [18]:
# effect of the instrument on the explanatory variable
lm6 = sm.ols(formula = "ed76 ~ nearc4 + exp + exp2 + black + reg76r + smsa76r + smsa66r + reg662 + reg663 + reg664 + reg665 + reg666 + reg667 + reg668 + reg669", data=x1).fit()
print(lm6.params['nearc4'])

0.3198989400914042


In [19]:
# IV estimate of returns to schooling
print(lm5.params['nearc4']/lm6.params['nearc4'])

0.13150382554546158


## 3.4.4 Matrix Algebra IV Estimates of Returns to Schooling

In [20]:
# the control variables, which are in X and also in Z
controls = ['black', 'reg76r', 'smsa76r', 'smsa66r', 'reg662', 'reg663',
            'reg664', 'reg665', 'reg666', 'reg667', 'reg668', 'reg669']

# we use numpy arrays (and not pandas dataframes)
# to pick the rows by their position in the bootstrap
y = np.array(x1['lwage76'])
X = np.array(x1[['ed76', 'exp', 'exp2'] + controls])

# three instruments:
# nearc4 for education, age for experience
# and age squared for experience squared
x1['age2'] = x1['age76']**2
Z1 = np.array(x1[['nearc4', 'age76', 'age2'] + controls])

In [21]:
res = lm_iv(y, X, Z1, Reps=1000)

# name the rows of the result table
res.index = ['intercept', 'ed76', 'exp', 'exp2'] + controls
res

,coef,sd,0.05,0.95
intercept,4.000811,0.821421,2.840394,4.861765
ed76,0.130631,0.070336,0.054600,0.231540
exp,0.059902,0.029251,0.010662,0.101289
exp2,-0.097993,0.150527,-0.310527,0.153370
black,-0.122945,0.097922,-0.228069,0.015676
reg76r,-0.142630,0.033795,-0.195709,-0.089567
smsa76r,0.083652,0.070235,-0.019880,0.165258
smsa66r,0.030333,0.024063,-0.007248,0.066230
reg662,0.075068,0.049250,-0.000537,0.147582
reg663,0.125467,0.044921,0.055377,0.193428


## 3.4.5 Concerns with Distance to College

In [22]:
# these are the variables we are interested in comparing.
tab_rows = ['ed76', 'exp', 'black', 'south66', 'smsa66r', 'reg76r', 'smsa76r']

# mean of each variable for each type
near = x1[x1['nearc4']==1]
not_near = x1[x1['nearc4']==0]

table_prox = pd.DataFrame()
table_prox['Near College'] = near[tab_rows].mean()
table_prox['Not Near College'] = not_near[tab_rows].mean()
table_prox

,Near College,Not Near College
ed76,13.527034,12.698015
exp,8.681929,9.229885
black,0.211885,0.280042
south66,0.328787,0.597701
smsa66r,0.799318,0.328109
reg76r,0.329274,0.563218
smsa76r,0.822211,0.478579


## 3.5.2 Test of Instrument Validity in Python

In [23]:
# second instrument for education: momdad14
# (both parents at home when the young man was 14)
Z2 = np.array(x1[['momdad14', 'age76', 'age2'] + controls])

We compare the two IV estimates of the returns to schooling on bootstrap samples.

In the book, the R code takes the element [2,1] of the estimate, with matrices that have no column of 1's. This element is the coefficient of exp. Here we add the column of 1's and we take the coefficient of ed76, which is the one described in the text. So the numbers are not the ones of the book.

In [24]:
# we add the column of 1's to the three matrices
# (lm_iv does it for us, but here the bootstrap is done by hand)
N = len(y)
X_1 = np.c_[np.ones(N), X]
Z1_1 = np.c_[np.ones(N), Z1]
Z2_1 = np.c_[np.ones(N), Z2]

# bootstrap
np.random.seed(123456789)
rows = []

for i in range(0, 1000):

    index_bs = np.random.randint(0, N, size=N)
    y_bs = y[index_bs]
    X_bs = X_1[index_bs, :]
    Z1_bs = Z1_1[index_bs, :]
    Z2_bs = Z2_1[index_bs, :]

    # IV estimates with the first and with the second instrument
    beta_1 = inv(Z1_bs.T.dot(X_bs)).dot(Z1_bs.T).dot(y_bs)
    beta_2 = inv(Z2_bs.T.dot(X_bs)).dot(Z2_bs.T).dot(y_bs)

    # position 0 is the intercept, position 1 is ed76
    # store the difference between the two estimates
    rows.append(beta_1[1] - beta_2[1])

bs_diff = pd.DataFrame(rows, columns=['difference'])

In [25]:
bs_diff.describe()

,difference
count,1000.000000
mean,0.015634
std,0.073139
min,-0.750217
25%,-0.025626
50%,0.009024
75%,0.049137
max,0.847465


In [26]:
# 90% confidence interval of the difference
bs_diff['difference'].quantile([0.05, 0.95])

0.05   -0.072784
0.95    0.120278
Name: difference, dtype: float64

## 3.6.4 LATE Estimates of Returns to Schooling

In [27]:
# college indicator: True with more than 12 years of education
# ed76 is the first column of X, so position 0 in Python
X2 = X[:, 0] > 12

# using college proximity as an instrument.
nearc4 = Z1[:, 0]
mu_y1 = np.mean(y[nearc4==1])
mu_y0 = np.mean(y[nearc4==0])

# the mean of a True/False variable gives the probability
p_11 = np.mean(X2[nearc4==1])
p_10 = np.mean(X2[nearc4==0])

# LATE, divide by 4 to get the per-year effect
# this allows comparison with the OLS estimates.
print(((mu_y1 - mu_y0)/(p_11 - p_10))/4)

0.31966789417922487


In [28]:
# using living with both parents as an instrument.
momdad14 = Z2[:, 0]
mu_y1 = np.mean(y[momdad14==1])
mu_y0 = np.mean(y[momdad14==0])
p_11 = np.mean(X2[momdad14==1])
p_10 = np.mean(X2[momdad14==0])
print(((mu_y1 - mu_y0)/(p_11 - p_10))/4)

0.17729674982778038
